# Swift 1.5 on Amazon SageMaker AI (AWS Marketplace)

Swift is UkisAI's reasoning model family, tuned to reach the same answers with far fewer reasoning tokens. This notebook deploys a Swift model package from AWS Marketplace, runs real-time and batch inference, and cleans up.

**Prerequisites:** subscribe to the Swift listing in AWS Marketplace, an IAM role with `AmazonSageMakerFullAccess`, and endpoint quota for the instance type you pick.

## 1. Set up
Paste the model package ARN for your Region from the listing's **Configure software** page.

In [ ]:
import json, boto3, sagemaker
from sagemaker import ModelPackage, get_execution_role

session = sagemaker.Session()
role = get_execution_role()
region = session.boto_region_name

# Swift 1.5 27B, or Swift 1.5 Flash Next
model_package_arn = "<model-package-arn-from-marketplace>"
instance_type = "ml.g7e.2xlarge"   # recommended; ml.g5.2xlarge also supported for the 27B


## 2. Deploy a real-time endpoint
Loading the weights takes a few minutes. The container exposes an OpenAI-compatible chat completions API on `/invocations`.

In [ ]:
model = ModelPackage(role=role, model_package_arn=model_package_arn, sagemaker_session=session)
endpoint_name = "swift-demo"
model.deploy(initial_instance_count=1, instance_type=instance_type, endpoint_name=endpoint_name,
             container_startup_health_check_timeout=1800, model_data_download_timeout=1800)


## 3. Chat completion
The request body follows the OpenAI chat completions format. The model's reasoning is returned in `reasoning_content` and the final answer in `content`.

In [ ]:
runtime = boto3.client("sagemaker-runtime")
body = json.load(open("data/realtime-input.json"))
resp = runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json", Body=json.dumps(body))
out = json.loads(resp["Body"].read())
msg = out["choices"][0]["message"]
print("reasoning tokens used:", out["usage"]["completion_tokens"])
print(msg["content"])


## 4. Streaming
Set `"stream": true` and use `invoke_endpoint_with_response_stream` to receive server-sent events.

In [ ]:
body["stream"] = True
stream = runtime.invoke_endpoint_with_response_stream(EndpointName=endpoint_name, ContentType="application/json", Body=json.dumps(body))
buf = b""
for event in stream["Body"]:
    buf += event["PayloadPart"]["Bytes"]
    *lines, buf = buf.split(b"\n")
    for line in lines:
        if line.startswith(b"data: ") and line != b"data: [DONE]":
            delta = json.loads(line[6:])["choices"][0]["delta"]
            print(delta.get("content") or "", end="", flush=True)


## 5. Batch transform
Each line of the input file is one chat request (JSON Lines).

In [ ]:
bucket = session.default_bucket()
input_uri = session.upload_data("data/batch-input.jsonl", bucket=bucket, key_prefix="swift-batch/input")
transformer = model.transformer(instance_count=1, instance_type="ml.g5.2xlarge", strategy="SingleRecord",
                                assemble_with="Line", accept="application/json",
                                output_path=f"s3://{bucket}/swift-batch/output")
transformer.transform(input_uri, content_type="application/jsonlines", split_type="Line")
transformer.wait()


## 6. Clean up
Delete the endpoint to stop instance and software charges.

In [ ]:
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)
model.delete_model()
